# Zoom (resimulation) cosmological simulations

Resolving one galaxy or cluster at high resolution in a full cosmological box is expensive &mdash;
you would have to refine the *whole* box. A **zoom** (or *resimulation*) avoids this: run a cheap
low-resolution box, pick an interesting object (here a massive halo), find the small **Lagrangian
region** its mass came from in the initial conditions, and regenerate the ICs with **extra
resolution only in that region**. Rerunning the box then resolves that one halo at high resolution
while the rest stays coarse (but still provides the correct large-scale tidal field).

This whole tutorial runs in a few minutes because we start from a deliberately **cheap** coarse box
($64^3$, `levelmin=6`) and add **3 levels** of resolution in the zoom region.

**You will need:** a working **MUSIC2** (IC generator), RAMSES built for cosmology
(`NDIM=3 MPI=1`), and `yt` + `yt_astro_analysis` for halo finding. Set the two paths below.

In [ ]:
import numpy as np, glob, os, subprocess
import yt
import matplotlib.pyplot as plt
from yt_astro_analysis.halo_analysis import HaloCatalog
yt.funcs.mylog.setLevel("ERROR")

MUSIC  = "/path/to/MUSIC2/build/MUSIC"      # <-- adapt
RAMSES = "/path/to/ramses/bin/ramses3d"     # <-- adapt (built with NDIM=3 MPI=1)
MPIRUN = "mpirun -np 8"                      # or "" for serial

## Step 1 &mdash; a cheap coarse run
We first generate a uniform $64^3$ (`levelmin=6`) dark-matter IC with MUSIC2 and run it to $z=0$
with a little AMR (`levelmax=8`). The `coarse_music.conf` and `coarse.nml` are provided next to
this notebook (a 400 Mpc/h box, Planck-like cosmology). RAMSES writes an output at the start
($z\simeq50$, used later for tracing) and at $z=0$.

In [ ]:
# generate the coarse ICs and run the coarse box (both are fast: seconds)
os.makedirs("coarse/ICs", exist_ok=True)
subprocess.run("cp coarse_music.conf coarse/ICs/music.conf && cd coarse/ICs && %s music.conf > music.log 2>&1" % MUSIC, shell=True)
subprocess.run("cp coarse.nml coarse/ && cd coarse && %s %s coarse.nml > run.log 2>&1" % (MPIRUN, RAMSES), shell=True)
print("coarse outputs:", sorted(os.path.basename(p) for p in glob.glob("coarse/output_*")))

## Step 2 &mdash; find the halos and pick one
Run a friends-of-friends finder on the $z=0$ snapshot and choose a massive halo **inside** the box
(so its Lagrangian region doesn't wrap around the periodic edges). With only $64^3$ particles even a
cluster is traced by $\sim\!100$ particles &mdash; enough to define the region, but note that a
*small* halo would have too few particles to trace reliably (a nice thing to experiment with).

In [ ]:
ds0 = yt.load("coarse/output_00002/info_00002.txt")
if not os.path.exists("coarse/Halos/info_00002/info_00002.0.h5"):
    HaloCatalog(data_ds=ds0, finder_method="fof", output_dir="coarse/Halos").create()
ad = yt.load("coarse/Halos/info_00002/info_00002.0.h5").all_data()

mass = ad["halos","particle_mass"].to("Msun").v
hpos = ad["halos","particle_position"].to("code_length").v
hrv  = ad["halos","virial_radius"].to("code_length").v
inside = np.all((hpos > 0.2) & (hpos < 0.8), axis=1)
k = np.where(inside)[0][np.argmax(mass[inside])]
halo_center, halo_rvir, halo_mass = hpos[k], float(hrv[k]), float(mass[k])
print("halo: M = %.2e Msun   center = %s   rvir = %.4f"
      % (halo_mass, np.round(halo_center, 3), halo_rvir))

## Step 3 &mdash; trace the halo to the initial conditions
Take the halo's particles at $z=0$ (within the virial radius), look up the **same particle IDs** in
the near-initial ($z\simeq50$) snapshot, and those positions are the **Lagrangian region**. Matching
by ID works regardless of the IC particle ordering. We then build a periodic-safe bounding box and
pad it.

In [ ]:
sp = ds0.sphere(ds0.arr(halo_center, "code_length"), (halo_rvir, "code_length"))
member_ids = np.array(sp["all", "particle_identity"].astype("int64"))

ds50 = yt.load("coarse/output_00001/info_00001.txt"); a50 = ds50.all_data()
ids50 = np.array(a50["all", "particle_identity"].astype("int64"))
pos50 = np.c_[a50["all","particle_position_x"].v, a50["all","particle_position_y"].v,
              a50["all","particle_position_z"].v]
lag = pos50[np.argsort(ids50)[np.searchsorted(np.sort(ids50), member_ids)]]

def circ_center(p):                         # periodic-safe mean
    a = 2*np.pi*p
    return (np.arctan2(np.sin(a).mean(), np.cos(a).mean()) / (2*np.pi)) % 1.0
center    = np.array([circ_center(lag[:, i]) for i in range(3)])
unwrapped = center + ((lag - center + 0.5) % 1.0 - 0.5)
extent    = unwrapped.max(0) - unwrapped.min(0)
ref_center = np.round(center, 4)
ref_extent = np.round(np.clip(extent * 1.5, 0.05, None), 4)   # pad by 50%
print("%d member particles" % len(member_ids))
print("=> MUSIC  ref_center =", ref_center.tolist(), "  ref_extent =", ref_extent.tolist())

The picture (made by the cell below): the $z=0$ halo (red) traces back to a compact patch in the
ICs (blue); the black box is the padded region we will refine.

![tracing the Lagrangian region](zoom_trace.png)

In [ ]:
p0 = np.c_[ds0.all_data()["all","particle_position_x"].v, ds0.all_data()["all","particle_position_y"].v]
s0 = np.c_[sp["all","particle_position_x"].v, sp["all","particle_position_y"].v]
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].scatter(p0[::8,0], p0[::8,1], s=0.5, c="0.8"); ax[0].scatter(s0[:,0], s0[:,1], s=6, c="r")
ax[0].set(title="z=0: chosen halo (red)", xlabel="x", ylabel="y", aspect="equal")
ax[1].scatter(pos50[::8,0], pos50[::8,1], s=0.5, c="0.8"); ax[1].scatter(lag[:,0], lag[:,1], s=6, c="b")
from matplotlib.patches import Rectangle
ax[1].add_patch(Rectangle((ref_center[0]-ref_extent[0]/2, ref_center[1]-ref_extent[1]/2),
                          ref_extent[0], ref_extent[1], fill=False, ec="k", lw=2))
ax[1].set(title="z=50 (ICs): Lagrangian patch", xlabel="x", ylabel="y", aspect="equal")
fig.tight_layout()

## Step 4 &mdash; generate the zoom ICs (+3 levels)
We take the coarse `music.conf` (same box, cosmology and **random seeds** &mdash; so the large-scale
structure is identical), raise `levelmax` to **9** (3 levels of extra particles in the zoom region),
and set `ref_center` / `ref_extent` to the Lagrangian region. MUSIC writes a nested grafic IC
(`level_006`&hellip;`level_009`).

In [ ]:
conf = open("coarse_music.conf").read()
conf = conf.replace("levelmax\t\t= 6", "levelmax\t\t= 9")
conf = conf.replace("ref_center\t\t= 0.5, 0.5, 0.5", "ref_center\t\t= %g, %g, %g" % tuple(ref_center))
conf = conf.replace("ref_extent\t\t= 0.2, 0.2, 0.2", "ref_extent\t\t= %g, %g, %g" % tuple(ref_extent))
os.makedirs("zoom/ICs", exist_ok=True)
open("zoom/ICs/music_zoom.conf", "w").write(conf)
subprocess.run("cd zoom/ICs && %s music_zoom.conf > music.log 2>&1" % MUSIC, shell=True)
print("levels:", sorted(os.path.basename(p) for p in glob.glob("zoom/ICs/level_*")))
# MUSIC recenters the zoom region to the box centre and shifts the whole domain:
shift_line = [l for l in open("zoom/ICs/music.log") if "shifted by" in l]
print(shift_line[0].strip() if shift_line else "(no shift reported)")

> **Gotcha:** MUSIC **shifts the whole domain** so the zoom region sits at the box centre (it prints
> *"Domain will be shifted by &hellip;"*). So in the zoom run the halo is **not** at its coarse
> coordinates any more &mdash; it's near the box centre. We account for this below by locating the halo
> from its high-resolution particles rather than assuming the old position.

## Step 5 &mdash; run the zoom
MUSIC also writes a starter `ramses.nml`. The important parts: one `initfile` per level, and
`&REFINE_PARAMS` with `mass_cut_refine` &mdash; this makes RAMSES refine **only where the
high-resolution (low-mass) particles are**, following the zoom region. We cap the AMR at
`levelmax=10` (one above the particle level, for force resolution in the dense halo) and ask for a
$z=0$ output. The provided `zoom.nml` already has these edits.

In [ ]:
subprocess.run("cp zoom.nml zoom/ && cd zoom && %s %s zoom.nml > zoom_run.log 2>&1" % (MPIRUN, RAMSES), shell=True)
print("zoom outputs:", sorted(os.path.basename(p) for p in glob.glob("zoom/output_*")))
print(open("zoom/zoom_run.log").readlines()[-1].strip())

## Step 6 &mdash; compare coarse vs zoom
The same halo is now sampled by **$8^3 = 512\times$ more particles** (3 levels), so its shape and
substructure appear. We locate the zoom halo from the densest clump of its high-resolution particles
(remember the domain shift), and check for **contamination** (coarse, massive particles leaking into
the halo &mdash; should be ~0% for a well-padded region).

In [ ]:
dz = yt.load("zoom/output_00002/info_00002.txt"); adz = dz.all_data()
mz = adz["all", "particle_mass"].to("Msun").v
hires = mz < mz.min() * 1.5                                  # the high-resolution particles
pz = np.c_[adz["all","particle_position_x"].v, adz["all","particle_position_y"].v,
           adz["all","particle_position_z"].v]
H, edges = np.histogramdd(pz[hires], bins=32)                # densest high-res clump = the halo
idx = np.unravel_index(H.argmax(), H.shape)
zoom_center = np.array([(edges[d][idx[d]] + edges[d][idx[d]+1]) / 2 for d in range(3)])

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
for a, ds, c, lab in [(ax[0], ds0, halo_center, "coarse (lmin=6)"),
                      (ax[1], dz, zoom_center, "zoom (+3 levels)")]:
    s = ds.sphere(ds.arr(c, "code_length"), (3*halo_rvir, "code_length"))
    a.scatter(s["all","particle_position_x"].v - c[0], s["all","particle_position_y"].v - c[1],
              s=3, c=np.log10(s["all","particle_mass"].to("Msun").v), cmap="viridis", vmin=10.5, vmax=13.6)
    a.set(title="%s: %d particles in 3 rvir" % (lab, len(s["all","particle_mass"])),
          xlabel="x - x_halo", ylabel="y - y_halo", aspect="equal")
fig.tight_layout()

m1 = dz.sphere(dz.arr(zoom_center,"code_length"), (halo_rvir,"code_length"))["all","particle_mass"].to("Msun").v
print("resolution gain: %.0fx finer particle mass (= %d levels)"
      % (mz.max()/mz.min(), round(np.log2((mz.max()/mz.min())**(1/3)))))
print("contamination within rvir: %.1f%%" % (100 * np.mean(m1 > mz.min()*1.5)))

![coarse vs zoom](zoom_compare.png)

The zoom panel resolves the halo with hundreds of times more particles, revealing its shape and
substructure, while the coarse panel shows a sparse blob &mdash; at a tiny fraction of the cost of
running the whole box at that resolution. Contamination inside the virial radius should be ~0%.

### Going further
- **More levels / smaller region.** Raise `levelmax` for higher resolution, or tighten the padded
  region (a convex-hull/ellipsoid region wastes less high-res volume). Watch cost and contamination.
- **A less massive halo.** With $64^3$ a small halo has very few particles, so the Lagrangian region
  is noisy &mdash; a good exercise in the limits of the method (raise the coarse `levelmin` to help).
- **Add gas.** Set `baryons = yes` in MUSIC and `hydro=.true.` in RAMSES to resimulate with gas.
- **Check the match.** The large-scale structure (and the halo) should be the same as the coarse run
  up to the domain shift, since the random seeds are identical.